In [ ]:
import googlemaps
import csv
import time
import re

# Initialize the Client with your API key
gmaps = googlemaps.Client(key='REDACTED_GOOGLE_MAPS_KEY')

input_filename = 'addresses.txt'
output_filename = 'address_sublocality_map.csv'

def extract_best_sublocality(components):
    """
    Extracts the most granular sublocality/neighborhood from Google Maps components.
    Prioritizes: neighborhood > sublocality > locality > admin area > borough fallback.
    Never returns the address as sublocality if not detected.
    """
    exclude = {"manhattan", "bronx", "brooklyn", "queens", "staten island"}
    neighborhood = None
    sublocality = None
    locality = None
    admin_area = None

    for comp in components:
        types = comp.get("types", [])
        name = comp.get("long_name", "").strip()
        lname = name.lower()
        if "neighborhood" in types and not neighborhood and lname not in exclude:
            neighborhood = name
        if ("sublocality" in types or "sublocality_level_1" in types) and not sublocality and lname not in exclude:
            sublocality = name
        if "locality" in types and not locality and lname not in exclude:
            locality = name
        if ("administrative_area_level_2" in types or "administrative_area_level_3" in types) and not admin_area and lname not in exclude:
            admin_area = name
    # Return most granular available, never fallback to address
    return neighborhood or sublocality or locality or admin_area

# Fallback ZIP-to-borough mapping
ZIP_TO_BOROUGH = {
    **{str(z): 'Manhattan' for z in range(10001, 10283)},
    **{str(z): 'Bronx' for z in [10451,10452,10453,10454,10455,10456,10457,10458,10459,10460,10461,10462,10463,10464,10465,10466,10467,10468,10469,10470,10471,10472,10473,10474,10475]},
    **{str(z): 'Brooklyn' for z in [11201,11203,11204,11205,11206,11207,11208,11209,11210,11211,11212,11213,11214,11215,11216,11217,11218,11219,11220,11221,11222,11223,11224,11225,11226,11228,11229,11230,11231,11232,11233,11234,11235,11236,11237,11238,11249]},
    **{str(z): 'Queens' for z in [11101,11102,11103,11104,11105,11106,11109,11354,11355,11356,11357,11358,11360,11361,11362,11363,11364,11365,11366,11367,11368,11369,11370,11371,11372,11373,11374,11375,11377,11378,11379,11385,11411,11412,11413,11414,11415,11416,11417,11418,11419,11420,11421,11422,11423,11426,11427,11428,11429,11430,11432,11433,11434,11435,11436,11691,11692,11693,11694,11695,11697]},
    **{str(z): 'Staten Island' for z in [10301,10302,10303,10304,10305,10306,10307,10308,10309,10310,10312,10314]}
}

with open(input_filename, "r") as infile, open(output_filename, "w", newline="") as outfile:
    writer = csv.writer(outfile)
    writer.writerow(["Original Address", "ZIP Code", "Sublocality", "Borough"])

    current_zip = None
    for line in infile:
        line = line.strip()
        if not line:
            continue

        if line.startswith("ZIPCODE:"):
            parts = line.split()
            if len(parts) > 1 and parts[1].isdigit():
                current_zip = parts[1]
            else:
                current_zip = None

        elif line.startswith("- "):
            address = line[2:].strip()
            if not address or not current_zip:
                continue

            try:
                geocode_result = gmaps.geocode(f"{address}, New York, NY {current_zip}")
                sublocality = None
                borough = ZIP_TO_BOROUGH.get(current_zip, "Unknown")
                if geocode_result:
                    components = geocode_result[0].get("address_components", [])
                    sublocality = extract_best_sublocality(components)
                # If still no sublocality, set to 'Unknown' (never fallback to address)
                writer.writerow([address, current_zip, sublocality if sublocality else "Unknown", borough])
            except Exception as e:
                print(f"Error geocoding {address}, ZIP: {current_zip}: {e}")
            time.sleep(0.1)

In [ ]:
# import googlemaps
# import csv
# import time
# import re

# # Initialize the Client with your API key
# gmaps = googlemaps.Client(key='REDACTED_GOOGLE_MAPS_KEY')

# input_filename = 'addresses.txt'
# output_filename = 'address_sublocality_map.csv'

# def extract_neighborhood(components):
#     """
#     Extracts the most relevant neighborhood/sublocality
#     Avoids borough-level names like Manhattan, Bronx, Brooklyn, Queens, Staten Island
#     """
#     neighborhood = None
#     sublocality = None
#     exclude = {"manhattan", "bronx", "brooklyn", "queens", "staten island"}

#     for comp in components:
#         types = comp.get("types", [])
#         name = comp.get("long_name", "").strip()

#         if "neighborhood" in types and not neighborhood:
#             if name.lower() not in exclude:
#                 neighborhood = name

#         if ("sublocality" in types or "sublocality_level_1" in types) and not sublocality:
#             if name.lower() not in exclude:
#                 sublocality = name

#     # Prioritize neighborhood over sublocality
#     return neighborhood or sublocality

# with open(input_filename, "r") as infile, open(output_filename, "w", newline="") as outfile:
#     writer = csv.writer(outfile)
#     writer.writerow(["Original Address", "ZIP Code", "Sublocality"])

#     current_zip = None
#     for line in infile:
#         line = line.strip()
#         if not line:
#             continue

#         if line.startswith("ZIPCODE:"):
#             parts = line.split()
#             if len(parts) > 1 and parts[1].isdigit():
#                 current_zip = parts[1]
#             else:
#                 current_zip = None

#         elif line.startswith("- "):
#             address = line[2:].strip()
#             if not address or not current_zip:
#                 continue

#             try:
#                 geocode_result = gmaps.geocode(f"{address}, New York, NY {current_zip}")
#                 sublocality = None
#                 if geocode_result:
#                     components = geocode_result[0].get("address_components", [])
#                     sublocality = extract_neighborhood(components)

#                 writer.writerow([address, current_zip, sublocality if sublocality else "Unknown"])

#             except Exception as e:
#                 print(f"Error geocoding {address}, ZIP: {current_zip}: {e}")

#             # Sleep to avoid API quota issues
#             time.sleep(0.1)


In [10]:
import pandas as pd

df = pd.read_csv('address_sublocality_map.csv')
unknown_df = df[df['Sublocality'] == 'Unknown'][['Original Address', 'ZIP Code']]
print('Addresses and ZIP codes with Unknown sublocality:')
display(unknown_df)


Addresses and ZIP codes with Unknown sublocality:


,Original Address,ZIP Code
0,"Regis Residence, 2 E 55th Street #803",10022
1,2 E 55th Street,10022
2,246 E 58th Street,10022
3,444 E 57th Street #2D,10022
4,2 Sutton Place S,10022
...,...,...
4401,16 W 40th Street #25B,10018
4402,400 5th Avenue Apt 57E,10018
4500,300 W 145th Street #2B,10039
4501,234 W 148th Street #6A,10039


In [11]:
df = pd.read_csv('address_sublocality_map.csv')
unique_zipcodes = df['ZIP Code'].dropna().unique()
print('Unique ZIP codes:')
print(sorted(unique_zipcodes))

Unique ZIP codes:
[10001, 10002, 10003, 10004, 10005, 10006, 10007, 10009, 10010, 10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10021, 10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031, 10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10044, 10065, 10069, 10075, 10128, 10280, 10282, 10301, 10302, 10303, 10304, 10305, 10306, 10307, 10308, 10309, 10310, 10312, 10314, 10451, 10452, 10453, 10454, 10455, 10456, 10457, 10458, 10459, 10460, 10461, 10462, 10463, 10464, 10465, 10466, 10467, 10468, 10469, 10470, 10471, 10472, 10473, 10474, 10475, 11001, 11004, 11005, 11040, 11101, 11102, 11103, 11104, 11105, 11106, 11109, 11201, 11203, 11204, 11205, 11206, 11207, 11208, 11209, 11210, 11211, 11212, 11213, 11214, 11215, 11216, 11217, 11218, 11219, 11220, 11221, 11222, 11223, 11224, 11225, 11226, 11228, 11229, 11230, 11231, 11232, 11233, 11234, 11235, 11236, 11237, 11238, 11249, 11354, 11355, 11356, 11357, 11358, 11360, 11361, 11362, 11363, 11364, 11365, 1